# Heart Disease Prediction
Predicting the presence of heart disease using the Cleveland UCI dataset.

**Stack:** Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-Learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Loading

In [ ]:
cols = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','target']
df = pd.read_csv('heart.csv')
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

print(df.shape)
df.head()

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
df.describe().round(2)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['target'].value_counts()
axes[0].pie(counts, labels=['No Disease','Heart Disease'], autopct='%1.1f%%',
            colors=['#66b3ff','#ff6b6b'], startangle=90, explode=(0.05, 0.05))
axes[0].set_title('Target Distribution')

sns.countplot(x='target', data=df, ax=axes[1], palette=['#66b3ff','#ff6b6b'])
axes[1].set_xticklabels(['No Disease', 'Heart Disease'])
axes[1].set_title('Class Count')
for p in axes[1].patches:
    axes[1].annotate(str(p.get_height()), (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for t, lbl, clr in [(0,'No Disease','#66b3ff'), (1,'Heart Disease','#ff6b6b')]:
    df[df['target']==t]['age'].plot.hist(ax=axes[0], alpha=0.6, color=clr, label=lbl, bins=20)
axes[0].set_title('Age Distribution by Disease Status')
axes[0].set_xlabel('Age')
axes[0].legend()

sex_target = df.groupby(['sex','target']).size().unstack()
sex_target.index = ['Female','Male']
sex_target.columns = ['No Disease','Heart Disease']
sex_target.plot(kind='bar', ax=axes[1], color=['#66b3ff','#ff6b6b'], edgecolor='white', rot=0)
axes[1].set_title('Sex vs Heart Disease')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(13, 9))
mask = np.triu(np.ones_like(df.corr(), dtype=bool))
sns.heatmap(df.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

print('Correlation with target:')
print(df.corr()['target'].drop('target').sort_values(key=abs, ascending=False).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cp_map = {0:'Typical Angina', 1:'Atypical Angina', 2:'Non-Anginal Pain', 3:'Asymptomatic'}
df['cp_label'] = df['cp'].map(cp_map)
sns.countplot(x='cp_label', hue='target', data=df, ax=axes[0], palette=['#66b3ff','#ff6b6b'])
axes[0].set_title('Chest Pain Type vs Heart Disease')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(['No Disease','Heart Disease'])
df.drop(columns='cp_label', inplace=True)

sc = axes[1].scatter(df['age'], df['thalach'], c=df['target'],
                     cmap='RdYlGn_r', alpha=0.7, edgecolors='white', s=60)
axes[1].set_title('Age vs Max Heart Rate')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Max Heart Rate')
plt.colorbar(sc, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
num_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for i, feat in enumerate(num_features):
    sns.boxplot(x='target', y=feat, data=df, ax=axes[i], palette=['#66b3ff','#ff6b6b'])
    axes[i].set_xticklabels(['Healthy','Disease'])
    axes[i].set_title(feat.upper())
    axes[i].set_xlabel('')
plt.suptitle('Numeric Features vs Heart Disease', y=1.02)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
df['ca'].fillna(df['ca'].median(), inplace=True)
df['thal'].fillna(df['thal'].median(), inplace=True)
print('Missing values remaining:', df.isnull().sum().sum())

In [ ]:
df['age_group']   = pd.cut(df['age'], bins=[0,40,50,60,100], labels=['<40','40-50','50-60','60+'])
df['high_chol']   = (df['chol'] > 240).astype(int)
df['high_bp']     = (df['trestbps'] > 130).astype(int)
df['low_thalach'] = (df['thalach'] < 120).astype(int)
df['risk_score']  = df['high_chol'] + df['high_bp'] + df['low_thalach'] + df['exang']

df[['age_group','high_chol','high_bp','low_thalach','risk_score']].head()

In [ ]:
risk_target = df.groupby(['risk_score','target']).size().unstack().fillna(0)
risk_target.columns = ['No Disease','Heart Disease']
risk_target.plot(kind='bar', color=['#66b3ff','#ff6b6b'], edgecolor='white', rot=0)
plt.title('Engineered Risk Score vs Heart Disease')
plt.xlabel('Risk Score (0–4)')
plt.tight_layout()
plt.show()

In [ ]:
df = pd.get_dummies(df, columns=['age_group','cp','thal','restecg'], drop_first=True)
print('Shape after encoding:', df.shape)

## 4. Preprocessing & Train-Test Split

In [ ]:
X = df.drop(columns='target')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 5. Model Training


In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_sc, y_train)

y_pred  = model.predict(X_test_sc)
y_proba = model.predict_proba(X_test_sc)[:, 1]

print(f'Train Accuracy : {model.score(X_train_sc, y_train)*100:.2f}%')
print(f'Test  Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'ROC-AUC        : {roc_auc_score(y_test, y_proba):.4f}')

## 6. Evaluation

In [ ]:
print(classification_report(y_test, y_pred, target_names=['No Disease','Heart Disease']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Disease','Heart Disease'],
            yticklabels=['No Disease','Heart Disease'],
            annot_kws={'size':16})
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#ff6b6b', lw=2.5, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1],'k--', lw=1.5)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#ff6b6b')
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_[0]})
coef_df = coef_df.sort_values('Coefficient', ascending=False)

colors = ['#ff6b6b' if c > 0 else '#66b3ff' for c in coef_df['Coefficient']]
plt.figure(figsize=(12, 7))
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white', height=0.7)
plt.axvline(x=0, color='black', linewidth=1)
plt.title('Feature Coefficients  (Red = Increases Risk | Blue = Decreases Risk)')
plt.tight_layout()
plt.show()

## 7. Predict on a New Patient

In [ ]:
sample = pd.DataFrame([np.zeros(X.shape[1])], columns=X.columns)
sample['age']         = 55
sample['sex']         = 1
sample['trestbps']    = 145
sample['chol']        = 250
sample['thalach']     = 110
sample['exang']       = 1
sample['oldpeak']     = 2.5
sample['fbs']         = 1
sample['high_chol']   = 1
sample['high_bp']     = 1
sample['low_thalach'] = 1
sample['risk_score']  = 4

pred  = model.predict(scaler.transform(sample))[0]
proba = model.predict_proba(scaler.transform(sample))[0][1]

print(f'Prediction : {"Heart Disease" if pred == 1 else "No Heart Disease"}')
print(f'Probability: {proba*100:.1f}%')